In [2]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

In [3]:
# Sample text datset
text="""
      deep learning is a part of artificial intelligence
      deep learning uses neural network
      artifficial intelligenceis transforming the world
      machine learning and deep learning aare important
      deep lstm models improve sequence learning
      natural language processing uses deep learnning
      """

In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)

Total words: 29


In [5]:
input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
      n_gram_sequence = token_list[:i+1]
      input_sequences.append(n_gram_sequence)

In [6]:
# padding sequence
max_seq_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))

In [7]:
# spilt , one hot encoding
x=input_sequences[:,:-1]
y=input_sequences[:,-1]
y=to_categorical(y,num_classes=total_words)

In [8]:
# build deep lstm models
model=Sequential()
model.add(Embedding(input_dim=total_words,output_dim=64,input_length=max_seq_len-1))
# first lstm layer
model.add(LSTM(128,return_sequences=True))
# second lstm layer
model.add(LSTM(128))
# output layer
model.add(Dense(total_words,activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [11]:
# train model
model.fit(x,y,epochs=200,verbose=1)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.0645 - loss: 3.3677
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.1613 - loss: 3.3597
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.1613 - loss: 3.3513
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - accuracy: 0.1613 - loss: 3.3415
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.1613 - loss: 3.3292
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.1613 - loss: 3.3135
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.1613 - loss: 3.2929
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.1613 - loss: 3.2661
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.1613 - loss: 3.2319
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.1613 - loss: 3.1911
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.1613 - loss: 3.1500
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.161

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 7, 64)          │         1,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 7, 128)         │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 707,993 (2.70 MB)

 Trainable params: 235,997 (921.86 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 471,996 (1.80 MB)

In [16]:
# predict next word
def predict_next_word(model,tokenizer,text,max_seq_len,n_words):
  for _ in range(n_words):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
    predicted = np.argmax(model.predict(token_list,verbose=0), axis=-1)[0]
    output_word = ""
    for word, index in tokenizer.word_index.items():
      if index == predicted:
           output_word=word
           break
    # append predicted word
    text +=" " + output_word
  return text

In [17]:
# test prediction
seed_text="deep learning"
predicted_sentence = predict_next_word(model,tokenizer,seed_text,max_seq_len,3)
print("\nInput :",seed_text)
print("Predicted Sentence :",predicted_sentence)


Input : deep learning
Predicted Sentence : deep learning uses neural network
